# LeetCode 3782 — Last Remaining Integer After Alternating Deletion Operations bằng Python

- Link: https://leetcode.com/problems/last-remaining-integer-after-alternating-deletion-operations/
- Độ khó: **Hard**
- Chủ đề: Toán, đệ quy, theo dõi **cấp số cộng** (đầu dãy, công sai, số phần tử), biến thể Josephus

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lastInteger(self, n: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra sau mỗi lượt xoá, dãy còn lại luôn là một **cấp số cộng** — chỉ cần lưu `(head, step, cnt)`.
2. Phân biệt chính xác "xoá mỗi số thứ hai **bắt đầu từ trái**" (giữ số đầu tiên) với bài *Elimination Game* (LC 390, xoá số đầu tiên).
3. Xác định khi nào `head` thay đổi ở lượt xoá **từ phải**: phụ thuộc tính chẵn lẻ của `cnt`.
4. Viết lời giải `O(log n)` cho `n ≤ 10^15` theo hai cách: lặp trên cấp số cộng và **đệ quy tương hỗ** `L(n)`, `R(n)`.
5. Kiểm chứng bằng mô phỏng trực tiếp trên list cho `n` nhỏ.

## 2. Đề bài & ví dụ

Viết các số `1..n` từ trái sang phải. Lần lượt áp dụng hai thao tác (bắt đầu bằng thao tác 1) cho tới khi còn một số:

- **Thao tác 1:** bắt đầu từ trái, xoá mỗi số thứ hai (số thứ 2, 4, 6, … tính từ trái).
- **Thao tác 2:** bắt đầu từ phải, xoá mỗi số thứ hai (số thứ 2, 4, 6, … tính từ phải).

Trả về số cuối cùng còn lại.

**Ví dụ 1:** `n = 8`: `[1..8] → [1,3,5,7] → [3,7] → [3]` ⇒ `3`.

**Ví dụ 2:** `n = 5`: `[1..5] → [1,3,5] → [1,5] → [1]` ⇒ `1`.

**Ví dụ 3:** `n = 1` ⇒ `1`.

**Ràng buộc:** `1 ≤ n ≤ 10^15`.

In [1]:
import random, time
from functools import lru_cache

examples = [(8, 3), (5, 1), (1, 1)]
for n, exp in examples:
    print("n =", n, "-> kỳ vọng", exp)

n = 8 -> kỳ vọng 3
n = 5 -> kỳ vọng 1
n = 1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Slicing với bước nhảy trong Python

- `a[::2]` giữ các phần tử chỉ số `0, 2, 4, …` (tức số thứ 1, 3, 5 tính từ trái) ⇒ đúng thao tác 1.
- `a[::-1][::2][::-1]` giữ số thứ 1, 3, 5 **tính từ phải** ⇒ đúng thao tác 2.

In [2]:
a = list(range(1, 9))
print("thao tác 1:", a[::2])
b = a[::2]
print("thao tác 2:", b[::-1][::2][::-1])

thao tác 1: [1, 3, 5, 7]
thao tác 2: [3, 7]


### 3.2. Cấp số cộng (arithmetic progression)

Một dãy `head, head+step, head+2·step, …` có `cnt` phần tử được mô tả đầy đủ bởi bộ ba `(head, step, cnt)`. Giữ lại các phần tử ở chỉ số chẵn hoặc lẻ của một cấp số cộng vẫn được một cấp số cộng với công sai `2·step`.

In [3]:
head, step, cnt = 3, 4, 5
ap = [head + i * step for i in range(cnt)]
print("cấp số:", ap, "| chỉ số chẵn:", ap[::2], "| chỉ số lẻ:", ap[1::2])

cấp số: [3, 7, 11, 15, 19] | chỉ số chẵn: [3, 11, 19] | chỉ số lẻ: [7, 15]


### 3.3. Số phần tử còn lại sau một lượt

Xoá mỗi số thứ hai từ một đầu giữ lại `⌈cnt/2⌉` phần tử (đầu bắt đầu luôn được giữ). Trong Python: `(cnt + 1) // 2`.

In [4]:
for cnt in range(1, 8):
    print(cnt, "->", (cnt + 1) // 2, "| kiểm tra:", len(list(range(cnt))[::2]))

1 -> 1 | kiểm tra: 1
2 -> 1 | kiểm tra: 1
3 -> 2 | kiểm tra: 2
4 -> 2 | kiểm tra: 2
5 -> 3 | kiểm tra: 3
6 -> 3 | kiểm tra: 3
7 -> 4 | kiểm tra: 4


### 3.4. Họ bài Josephus và kỹ thuật "chuẩn hoá"

Trong bài Josephus kinh điển (n người đứng vòng tròn, cứ đếm tới `k` thì loại), ta có công thức đệ quy `J(n) = (J(n−1) + k) mod n`: sau khi loại một người, **đánh số lại** những người còn lại thành `0..n−2` và giải bài nhỏ hơn, rồi **ánh xạ ngược** vị trí về dãy cũ.

Bài này dùng đúng tinh thần đó nhưng mỗi bước loại **một nửa**: sau một lượt, dãy còn lại (dù là `1,3,5,…` hay `2,4,6,…`) được đánh số lại thành `1..⌈m/2⌉`, và ánh xạ ngược là hàm tuyến tính `x ↦ 2x − 1` hoặc `x ↦ 2x`. Vì số phần tử giảm một nửa mỗi bước, độ sâu đệ quy chỉ là `O(log n)` thay vì `O(n)` như Josephus.

In [5]:
# Josephus kinh điển (k = 2, vòng tròn) để so sánh phong cách đệ quy
def josephus(n, k=2):
    pos = 0
    for m in range(2, n + 1):
        pos = (pos + k) % m
    return pos + 1

print("Josephus k=2, n=1..10:", [josephus(n) for n in range(1, 11)])

Josephus k=2, n=1..10: [1, 1, 3, 1, 3, 5, 7, 1, 3, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Không thể mô phỏng với `n = 10^15`, nhưng dãy còn lại **luôn đều đặn**:

- Thao tác 1 (từ trái): giữ chỉ số `0, 2, 4, …` ⇒ `head` **không đổi**, `step` gấp đôi.
- Thao tác 2 (từ phải): giữ chỉ số `cnt−1, cnt−3, …`.
  - Nếu `cnt` **lẻ**: chỉ số `0` cũng được giữ ⇒ `head` không đổi.
  - Nếu `cnt` **chẵn**: chỉ số `0` bị xoá, phần tử đầu mới là chỉ số `1` ⇒ `head += step`.

Mỗi lượt `cnt` giảm còn một nửa ⇒ khoảng `log₂ n ≈ 50` lượt.

Kiểm tra `n = 8`: `(1,1,8)` → T1 → `(1,2,4)` → T2 (`cnt=4` chẵn) → `(3,4,2)` → T1 → `(3,8,1)` ⇒ `3`. ✔

## 5. Cách 1 — Brute Force: mô phỏng trên list

In [6]:
def brute(n):
    a = list(range(1, n + 1))
    left = True
    while len(a) > 1:
        if left:
            a = a[::2]
        else:
            a = a[::-1][::2][::-1]
        left = not left
    return a[0]

for n, exp in examples:
    print(n, "->", brute(n), "kỳ vọng", exp)

8 -> 3 kỳ vọng 3
5 -> 1 kỳ vọng 1
1 -> 1 kỳ vọng 1


**Nút thắt:** tổng chi phí `n + n/2 + n/4 + … = O(n)` thời gian và bộ nhớ — với `10^15` là bất khả thi (thậm chí không tạo nổi list).

## 6. Ý tưởng thuật toán / bước đột phá

Thay vì giữ cả dãy, chỉ giữ **mô tả** của nó: `(head, step, cnt)`. Mỗi lượt cập nhật `O(1)`, số lượt `O(log n)`.

Một cách nhìn khác tương đương: **đệ quy tương hỗ**. Gọi `L(m)` = kết quả khi bắt đầu bằng thao tác 1 trên dãy `1..m`, `R(m)` = kết quả khi bắt đầu bằng thao tác 2.

- Sau thao tác 1 trên `1..m`, còn `1, 3, 5, …` = `2i − 1` với `i = 1..⌈m/2⌉`, rồi tới lượt thao tác 2 ⇒ `L(m) = 2·R(⌈m/2⌉) − 1`.
- Sau thao tác 2 trên `1..m`: nếu `m` lẻ còn `1, 3, …` (= `2i − 1`); nếu `m` chẵn còn `2, 4, …` (= `2i`) ⇒ `R(m) = 2·L(⌈m/2⌉) − (m mod 2)`.
- `L(1) = R(1) = 1`.

## 7. Các cách tiếp theo

- **Cách 2:** mô phỏng bằng `deque`/slicing nhưng tiết kiệm hơn (vẫn `O(n)`) — để thấy giới hạn.
- **Cách 3:** theo dõi cấp số cộng `(head, step, cnt)` — `O(log n)` (lời giải nộp).
- **Cách 4:** đệ quy tương hỗ `L`, `R` — `O(log n)`.
- **Cách 5:** viết đáp án dạng tổng: `ans = 1 + Σ step` trên các lượt thao tác 2 có `cnt` chẵn.

### 7.1. Cách 2 — Mô phỏng chỉ theo chỉ số (range object)

Dùng `range` của Python (lưu `start, stop, step` — chính là cấp số cộng!) và slicing trên `range` vẫn trả về `range`. Đây là cầu nối tự nhiên giữa Brute Force và Cách 3: Python tự làm việc "giữ mô tả" giúp ta.

In [7]:
def range_sim(n):
    r = range(1, n + 1)
    left = True
    while len(r) > 1:
        r = r[::2] if left else r[::-1][::2][::-1]
        left = not left
    return r[0]

print([range_sim(n) for n, _ in examples], "| n = 10^15:", range_sim(10**15))
print("range sau một lượt:", range(1, 9)[::2], "| sau hai lượt:", range(1, 9)[::2][::-1][::2][::-1])

[3, 1, 1] | n = 10^15: 712666616310443
range sau một lượt: range(1, 9, 2) | sau hai lượt: range(3, 11, 4)


**Độ phức tạp:** mỗi slicing trên `range` là `O(1)` ⇒ `O(log n)`. Rất gọn, nhưng nên hiểu bản chất (Cách 3) để viết được bằng ngôn ngữ khác.

### 7.2. Cách 3 — Theo dõi cấp số cộng (lời giải chính)

In [8]:
def ap_track(n):
    head, step, cnt = 1, 1, n
    left = True
    while cnt > 1:
        if not left and cnt % 2 == 0:   # xoá từ phải, cnt chẵn: phần tử đầu bị xoá
            head += step
        step *= 2
        cnt = (cnt + 1) // 2
        left = not left
    return head

print([ap_track(n) for n, _ in examples])

[3, 1, 1]


### 7.3. Cách 4 — Đệ quy tương hỗ

In [9]:
def L(m):
    return 1 if m == 1 else 2 * R((m + 1) // 2) - 1

def R(m):
    return 1 if m == 1 else 2 * L((m + 1) // 2) - (m % 2)

print([L(n) for n, _ in examples], "| độ sâu đệ quy với 10^15 ≈", (10**15).bit_length())

[3, 1, 1] | độ sâu đệ quy với 10^15 ≈ 50


### 7.4. Cách 5 — Dạng tổng

Từ Cách 3: `head` chỉ tăng ở các lượt thao tác 2 với `cnt` chẵn, mỗi lần tăng đúng `step` hiện tại (`= 2^{số lượt đã qua}`). Vậy `ans = 1 + Σ 2^t` trên các lượt `t` lẻ (0-indexed) mà `cnt_t` chẵn — đáp án là `1 +` một số mà các bit bật chỉ nằm ở vị trí lẻ.

In [10]:
def as_sum(n):
    cnt, t, extra = n, 0, 0
    while cnt > 1:
        if t % 2 == 1 and cnt % 2 == 0:
            extra |= 1 << t
        cnt = (cnt + 1) // 2; t += 1
    return 1 + extra

for n in [8, 5, 1, 100, 12345]:
    print(n, as_sum(n), "| bit của ans-1:", bin(as_sum(n) - 1))

8 3 | bit của ans-1: 0b10
5 1 | bit của ans-1: 0b0
1 1 | bit của ans-1: 0b0
100 35 | bit của ans-1: 0b100010
12345 8233 | bit của ans-1: 0b10000000101000


### 7.5. Quan sát bảng đáp án

In đáp án cho `n = 1..40` để thấy cấu trúc: đáp án **không đơn điệu**, giữ nguyên trên các đoạn và "nhảy" khi tính chẵn lẻ của `cnt` ở một lượt thao tác 2 thay đổi. Đáp án luôn có dạng `1 + (số chỉ bật bit ở vị trí lẻ)`, ví dụ `1, 3, 9, 11, 33, …` (`ans − 1 ∈ {0, 2, 8, 10, 32, …}`).

In [11]:
table = [(n, Solution().lastInteger(n) if 'Solution' in globals() else ap_track(n)) for n in range(1, 41)]
for i in range(0, 40, 10):
    print("  ".join(f"{n:>2}:{v:<3}" for n, v in table[i:i + 10]))
vals = sorted({ap_track(n) - 1 for n in range(1, 5000)})
print("các giá trị ans-1 xuất hiện (n < 5000):", vals[:12])
assert all((v & 0x5555555555555555) == 0 for v in vals)   # không có bit ở vị trí chẵn
print("Mọi ans-1 chỉ có bit ở vị trí lẻ: đúng")

 1:1     2:1     3:3     4:3     5:1     6:1     7:3     8:3     9:9    10:9  
11:11   12:11   13:9    14:9    15:11   16:11   17:1    18:1    19:3    20:3  
21:1    22:1    23:3    24:3    25:9    26:9    27:11   28:11   29:9    30:9  
31:11   32:11   33:33   34:33   35:35   36:35   37:33   38:33   39:35   40:35 
các giá trị ans-1 xuất hiện (n < 5000): [0, 2, 8, 10, 32, 34, 40, 42, 128, 130, 136, 138]
Mọi ans-1 chỉ có bit ở vị trí lẻ: đúng


## 8. TRACE từng bước

In [12]:
def trace_ap(n, show_list=True):
    head, step, cnt, left, t = 1, 1, n, True, 0
    print(f"bắt đầu: head={head}, step={step}, cnt={cnt}", "|", [head + i * step for i in range(cnt)] if show_list else "")
    while cnt > 1:
        op = "T1 (trái)" if left else "T2 (phải)"
        moved = (not left and cnt % 2 == 0)
        if moved:
            head += step
        step *= 2; cnt = (cnt + 1) // 2; t += 1
        note = "cnt chẵn ⇒ head += step cũ" if moved else ""
        print(f"lượt {t} {op}: head={head}, step={step}, cnt={cnt}", "|", [head + i * step for i in range(cnt)] if show_list else "", note)
        left = not left
    print("đáp án:", head)

trace_ap(8)

bắt đầu: head=1, step=1, cnt=8 | [1, 2, 3, 4, 5, 6, 7, 8]
lượt 1 T1 (trái): head=1, step=2, cnt=4 | [1, 3, 5, 7] 
lượt 2 T2 (phải): head=3, step=4, cnt=2 | [3, 7] cnt chẵn ⇒ head += step cũ
lượt 3 T1 (trái): head=3, step=8, cnt=1 | [3] 
đáp án: 3


In [13]:
trace_ap(10)
print("brute:", brute(10))

bắt đầu: head=1, step=1, cnt=10 | [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
lượt 1 T1 (trái): head=1, step=2, cnt=5 | [1, 3, 5, 7, 9] 
lượt 2 T2 (phải): head=1, step=4, cnt=3 | [1, 5, 9] 
lượt 3 T1 (trái): head=1, step=8, cnt=2 | [1, 9] 
lượt 4 T2 (phải): head=9, step=16, cnt=1 | [9] cnt chẵn ⇒ head += step cũ
đáp án: 9
brute: 9


In [14]:
trace_ap(10**15, show_list=False)

bắt đầu: head=1, step=1, cnt=1000000000000000 | 
lượt 1 T1 (trái): head=1, step=2, cnt=500000000000000 |  
lượt 2 T2 (phải): head=3, step=4, cnt=250000000000000 |  cnt chẵn ⇒ head += step cũ
lượt 3 T1 (trái): head=3, step=8, cnt=125000000000000 |  
lượt 4 T2 (phải): head=11, step=16, cnt=62500000000000 |  cnt chẵn ⇒ head += step cũ
lượt 5 T1 (trái): head=11, step=32, cnt=31250000000000 |  
lượt 6 T2 (phải): head=43, step=64, cnt=15625000000000 |  cnt chẵn ⇒ head += step cũ
lượt 7 T1 (trái): head=43, step=128, cnt=7812500000000 |  
lượt 8 T2 (phải): head=171, step=256, cnt=3906250000000 |  cnt chẵn ⇒ head += step cũ
lượt 9 T1 (trái): head=171, step=512, cnt=1953125000000 |  
lượt 10 T2 (phải): head=683, step=1024, cnt=976562500000 |  cnt chẵn ⇒ head += step cũ
lượt 11 T1 (trái): head=683, step=2048, cnt=488281250000 |  
lượt 12 T2 (phải): head=2731, step=4096, cnt=244140625000 |  cnt chẵn ⇒ head += step cũ
lượt 13 T1 (trái): head=2731, step=8192, cnt=122070312500 |  
lượt 14 T2 (phải): 

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def lastInteger(self, n: int) -> int:
        head, step, cnt = 1, 1, n
        left = True
        while cnt > 1:
            if not left and cnt % 2 == 0:
                head += step
            step <<= 1
            cnt = (cnt + 1) >> 1
            left = not left
        return head

sol = Solution()
for n, exp in examples:
    got = sol.lastInteger(n)
    print(n, "->", got, "✔" if got == exp else "✘")

8 -> 3 ✔
5 -> 1 ✔
1 -> 1 ✔


## 10. Vì sao thuật toán đúng

**Bất biến:** trước mỗi lượt, tập số còn lại là `{head + i·step : 0 ≤ i < cnt}` theo thứ tự tăng dần.

- **Khởi tạo:** `head = 1, step = 1, cnt = n` — đúng là `1..n`.
- **Thao tác 1:** giữ chỉ số `0, 2, …` ⇒ phần tử đầu vẫn là `head`, công sai `2·step`, số lượng `⌈cnt/2⌉`.
- **Thao tác 2:** giữ chỉ số `cnt−1, cnt−3, …` ⇒ cùng tính chẵn lẻ với `cnt−1`. Nếu `cnt` lẻ, `cnt−1` chẵn ⇒ chỉ số `0` được giữ, `head` không đổi; nếu `cnt` chẵn ⇒ chỉ số nhỏ nhất được giữ là `1`, `head` mới `= head + step`. Công sai `2·step`, số lượng `⌈cnt/2⌉`.
- **Kết thúc:** khi `cnt = 1`, dãy chỉ còn `head`.

Vòng lặp chạy `⌈log₂ n⌉` lần vì `cnt` giảm còn `⌈cnt/2⌉`. Đệ quy `L/R` là cùng lập luận nhưng "chuẩn hoá" dãy về `1..m` sau mỗi lượt.

**Kiểm chứng ánh xạ của đệ quy.** Với `L(m) = 2·R(⌈m/2⌉) − 1`: sau thao tác 1, phần tử thứ `i` (1-indexed) của dãy mới là `2i − 1`. Nếu bài toán con (bắt đầu bằng thao tác 2 trên `1..⌈m/2⌉`) trả về vị trí `p`, thì giá trị thật là `2p − 1`. Tương tự với `R`: dãy mới là `2i − (m mod 2)` vì khi `m` lẻ giữ các số lẻ, khi `m` chẵn giữ các số chẵn. Hai hàm luân phiên vì các thao tác luân phiên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng list | `O(n)` | `O(n)` |
| 2. Slicing `range` | `O(log n)` | `O(1)` |
| 3. Cấp số cộng | `O(log n)` | `O(1)` |
| 4. Đệ quy `L/R` | `O(log n)` | `O(log n)` (ngăn xếp) |
| 5. Dạng tổng | `O(log n)` | `O(1)` |

## 12. Bộ test

In [16]:
def all_fast(n):
    return [range_sim(n), ap_track(n), L(n), as_sum(n), Solution().lastInteger(n)]

for n, exp in examples:
    assert brute(n) == exp and all(v == exp for v in all_fast(n))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 toàn bộ n nhỏ: so với mô phỏng
for n in range(1, 3001):
    b = brute(n)
    assert all(v == b for v in all_fast(n)), (n, b, all_fast(n))
print("n = 1..3000: OK")

n = 1..3000: OK


In [18]:
# 12.3 n ngẫu nhiên lớn: các cách nhanh phải khớp nhau
rng = random.Random(3782)
for _ in range(2000):
    n = rng.randint(1, 10**15)
    assert len(set(all_fast(n))) == 1
t0 = time.time()
for _ in range(10**4):
    Solution().lastInteger(10**15)
print("2000 n lớn khớp nhau | 10^4 lần gọi với n = 10^15 mất", round(time.time() - t0, 3), "s")

2000 n lớn khớp nhau | 10^4 lần gọi với n = 10^15 mất 0.068 s


In [19]:
# 12.4 các ca biên đáng chú ý
for n in [1, 2, 3, 4, 2**10, 2**10 + 1, 2**50 - 1, 10**15]:
    print(n, "->", Solution().lastInteger(n))
assert Solution().lastInteger(2) == 1 == brute(2)
assert Solution().lastInteger(4) == brute(4)

1 -> 1
2 -> 1
3 -> 3
4 -> 3
1024 -> 683
1025 -> 1
1125899906842623 -> 750599937895083
1000000000000000 -> 712666616310443


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm với *Elimination Game* (LC 390): xoá luôn số đầu tiên

LC 390 xoá số thứ **1**, 3, 5…; ở đây xoá số thứ **2**, 4, 6… (giữ số đầu tiên tính từ phía bắt đầu).

In [20]:
def wrong_lc390(n):                       # SAI: logic của LC 390
    head, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            head += step
        step *= 2; cnt //= 2; left = not left
    return head

print("sai:", [wrong_lc390(n) for n, _ in examples], "| đúng:", [e for _, e in examples])

sai: [6, 2, 1] | đúng: [3, 1, 1]


### Lỗi 2 — Dùng `cnt // 2` thay vì `⌈cnt/2⌉`

Phía bắt đầu luôn được giữ, nên số còn lại là `(cnt + 1) // 2`.

In [21]:
def wrong_floor(n):
    head, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if not left and cnt % 2 == 0: head += step
        step *= 2; cnt //= 2; left = not left            # SAI
    return head

bad = [n for n in range(1, 200) if wrong_floor(n) != brute(n)]
print("số n sai trong 1..199:", len(bad), "| ví dụ:", bad[:5], "| n=5: sai", wrong_floor(5), "đúng", brute(5))

số n sai trong 1..199: 188 | ví dụ: [3, 5, 7, 9, 10] | n=5: sai 3 đúng 1


### Lỗi 3 — Đảo điều kiện chẵn lẻ ở thao tác 2

Khi `cnt` **chẵn** thì phần tử đầu bị xoá (không phải khi lẻ).

In [22]:
def wrong_parity(n):
    head, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if not left and cnt % 2 == 1: head += step       # SAI
        step *= 2; cnt = (cnt + 1) // 2; left = not left
    return head

print("n = 8: sai", wrong_parity(8), "| đúng", brute(8))

n = 8: sai 1 | đúng 3


### Lỗi 4 — Mô phỏng thật với `n` lớn

`list(range(1, 10**15 + 1))` cần khoảng `8·10^15` byte — tràn bộ nhớ ngay. Luôn nhìn ràng buộc trước khi mô phỏng.

In [23]:
print("ước lượng bộ nhớ list 10^15 phần tử (TB):", 8 * 10**15 / 1e12)

ước lượng bộ nhớ list 10^15 phần tử (TB): 8000.0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xoá cách một" lặp lại ⇒ dãy còn lại là **cấp số cộng** ⇒ lưu `(head, step, cnt)`.
- Câu hỏi then chốt mỗi lượt: **phần tử đầu có bị xoá không?** Phụ thuộc hướng và tính chẵn lẻ của `cnt`.
- Đệ quy "chuẩn hoá về `1..m`" là kỹ thuật chung của họ Josephus.
- `range` trong Python hỗ trợ slicing `O(1)` — mẹo cực nhanh để kiểm chứng.

## 15. Biến thể của bài

1. **LC 390 Elimination Game:** xoá số đầu tiên rồi cách một.
2. **Xoá mỗi số thứ `k`** (Josephus): công thức `J(n,k)`.
3. **Luôn xoá từ trái:** đáp án luôn là `1`.
4. **Bắt đầu bằng thao tác 2:** đáp án là `R(n)`.
5. **Hỏi thứ tự bị xoá của một số `x`**: theo dõi vị trí `x` trong cấp số cộng.

In [24]:
# Biến thể 4: bắt đầu bằng thao tác 2 — so với mô phỏng
def brute_from_right(n):
    a = list(range(1, n + 1)); left = False
    while len(a) > 1:
        a = a[::2] if left else a[::-1][::2][::-1]; left = not left
    return a[0]

assert all(R(n) == brute_from_right(n) for n in range(1, 2000))
print("Biến thể 4 (R(n)) khớp mô phỏng; ví dụ:", [R(n) for n in range(1, 11)])

Biến thể 4 (R(n)) khớp mô phỏng; ví dụ: [1, 2, 1, 2, 5, 6, 5, 6, 1, 2]


In [25]:
# Biến thể 5: x bị xoá ở lượt nào (0 nếu là số cuối cùng)
def removal_round(n, x):
    head, step, cnt, left, t = 1, 1, n, True, 0
    while cnt > 1:
        i = (x - head) // step                       # chỉ số của x trong cấp số hiện tại
        keep = (i % 2 == 0) if left else ((cnt - 1 - i) % 2 == 0)
        t += 1
        if not keep: return t
        if not left and cnt % 2 == 0: head += step
        step *= 2; cnt = (cnt + 1) // 2; left = not left
    return 0

print([removal_round(8, x) for x in range(1, 9)])

[2, 1, 0, 1, 2, 1, 3, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng list | Làm đúng như đề | `O(n)` | `O(n)` | Kiểm chứng |
| 2. Slicing `range` | Python giữ cấp số giúp | `O(log n)` | `O(1)` | Mẹo Python |
| 3. Cấp số cộng | `(head, step, cnt)` | `O(log n)` | `O(1)` | **Lời giải chuẩn** |
| 4. Đệ quy `L/R` | Chuẩn hoá về `1..m` | `O(log n)` | `O(log n)` | Chứng minh, biến thể |
| 5. Dạng tổng | Bit ở lượt lẻ | `O(log n)` | `O(1)` | Hiểu cấu trúc đáp án |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `lastInteger(6)` = ?

**Bài 2 (tính tay).** Trace `(head, step, cnt)` cho `n = 12`.

**Bài 3 (điền TODO).** Viết `elimination_390(n)` cho LC 390 (xoá số đầu từ trái, rồi từ phải, …).

**Bài 4 (tự viết lại).** Viết lại `L/R` thành một hàm duy nhất `f(m, left)`.

**Bài 5 (mini challenge).** Đếm số `n ≤ N` mà `lastInteger(n) = 1` với `N = 10^4` (gợi ý: dùng dạng tổng — đáp án `1` ⇔ không lượt thao tác 2 nào có `cnt` chẵn).

In [26]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def elimination_390(n):
    # TODO: như Cách 3 nhưng head thay đổi khi đi từ trái, hoặc từ phải với cnt lẻ
    pass

In [27]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_ones(N):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1..6] → [1,3,5]`. (2) Từ phải, `cnt = 3` lẻ ⇒ giữ `1, 5`. (3) Từ trái ⇒ `1`.

**Bài 2.** (1) `(1,1,12) → (1,2,6)`. (2) `cnt = 6` chẵn ⇒ `(3,4,3)`. (3) `→ (3,8,2) →` từ phải, `cnt = 2` chẵn ⇒ `(11,16,1)` ⇒ `11`.

**Bài 3.** (1) Từ trái: luôn xoá đầu ⇒ `head += step`. (2) Từ phải: xoá đầu khi `cnt` lẻ. (3) `cnt //= 2`.

**Bài 4.** (1) Tham số `left`. (2) `f(m, True) = 2·f(⌈m/2⌉, False) − 1`. (3) `f(m, False) = 2·f(⌈m/2⌉, True) − m%2`.

**Bài 5.** (1) Duyệt `n` và gọi lời giải. (2) Hoặc dùng `as_sum`. (3) Đối chiếu hai cách.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
assert Solution().lastInteger(6) == 1 == brute(6)
assert Solution().lastInteger(12) == 11 == brute(12)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [29]:
# Bài 3: LC 390
def elimination_390_sol(n):
    head, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            head += step
        step *= 2; cnt //= 2; left = not left
    return head

def brute_390(n):
    a = list(range(1, n + 1)); left = True
    while len(a) > 1:
        a = a[1::2] if left else a[::-1][1::2][::-1]; left = not left
    return a[0]

assert all(elimination_390_sol(n) == brute_390(n) for n in range(1, 2000))
print("Bài 3 OK; LC390(9) =", elimination_390_sol(9))

Bài 3 OK; LC390(9) = 6


In [30]:
# Bài 4, 5
@lru_cache(None)
def f(m, left):
    if m == 1: return 1
    h = (m + 1) // 2
    return 2 * f(h, False) - 1 if left else 2 * f(h, True) - (m % 2)

assert all(f(n, True) == Solution().lastInteger(n) for n in range(1, 5000))
c1 = sum(1 for n in range(1, 10**4 + 1) if Solution().lastInteger(n) == 1)
c2 = sum(1 for n in range(1, 10**4 + 1) if as_sum(n) == 1)
assert c1 == c2
print("Bài 4 OK | Bài 5: có", c1, "số n ≤ 10^4 cho đáp án 1")

Bài 4 OK | Bài 5: có 128 số n ≤ 10^4 cho đáp án 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
head, step, cnt, left = 1, 1, n, True
while cnt > 1:
    if not left and cnt % 2 == 0:   # từ phải + cnt chẵn ⇒ phần tử đầu bị xoá
        head += step
    step *= 2
    cnt = (cnt + 1) // 2
    left = not left
return head
```

**Checklist:**

- [ ] Đọc kỹ: giữ hay xoá phần tử đầu tiên (khác LC 390)?
- [ ] Số còn lại là `⌈cnt/2⌉`?
- [ ] `head` chỉ đổi ở thao tác 2 khi `cnt` chẵn?
- [ ] Không mô phỏng với `n` lớn?
- [ ] Đã so với mô phỏng list cho `n` nhỏ?